In [1]:
import os

In [2]:
%pwd

'c:\\projects\\SellWise\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\projects\\SellWise'

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict

@dataclass(frozen=True)
class NonRecursiveTrainerConfig:
    root_dir: Path
    processed_data_dir: Path
    models_dir: Path
    first_day: int
    cv: str
    early_stopping_rounds: int
    stores: List[str]
    cats: List[str]
    depts: List[str]
    lgb_params: Dict
    validation: Dict
    grid2_colnm: List[str]
    grid3_colnm: List[str]
    lag_colnm: List[str]
    remove_feature_by_level: Dict
    mean_enc_by_level: Dict

In [6]:
from SellWise.constants import *
from SellWise.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([(self.config.artifacts_root)])

    def get_nonrecursive_trainer_config(self) -> NonRecursiveTrainerConfig:
        config = self.config.fleet_trainer
        params = self.params.NONRECURSIVE_TRAINING
        lgb_params = self.params.LIGHTGBM

        create_directories([config.models_dir])

        return NonRecursiveTrainerConfig(
            root_dir=config.root_dir,
            processed_data_dir=config.processed_data_dir,
            models_dir=config.models_dir,
            first_day=params.first_day,
            cv=params.cv,
            early_stopping_rounds=params.early_stopping_rounds,
            stores=list(params.stores),
            cats=list(params.cats),
            depts=list(params.depts),
            lgb_params=dict(lgb_params),
            validation={k: list(v) for k, v in params.validation.items()},
            grid2_colnm=list(params.grid2_colnm),
            grid3_colnm=list(params.grid3_colnm),
            lag_colnm=list(params.lag_colnm),
            remove_feature_by_level={k: list(v) for k, v in params.remove_feature_by_level.items()},
            mean_enc_by_level={k: list(v) for k, v in params.mean_enc_by_level.items()}
        )

In [8]:
import gc
import pickle
import numpy as np, pandas as pd
import lightgbm as lgb
import warnings
from SellWise import logger

warnings.filterwarnings('ignore')

In [9]:
def reduce_mem_usage(df, verbose=False):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        logger.info('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(
            end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

In [10]:
class NonRecursiveTrainer:
    """
    Unifies A1's 3 non-recursive training notebooks (2-1 store, 2-2
    store+cat, 2-3 store+dept). Column lists, validation windows,
    per-level remove_feature/mean_enc sets all come from config/params.yaml
    (not hardcoded), and the level is passed explicitly as a string rather
    than inferred from column existence -- store_id/cat_id/dept_id columns
    are always present post-concat regardless of grouping level, so
    inferring level from "is this column present" is unreliable; the level
    argument is the single source of truth here.
    """

    def __init__(self, config: NonRecursiveTrainerConfig):
        self.config = config

    def _load_grid(self):
        p = self.config.processed_data_dir
        grid_1 = pd.read_pickle(f"{p}/grid_part_1.pkl")
        grid_2 = pd.read_pickle(f"{p}/grid_part_2.pkl")[self.config.grid2_colnm]
        grid_3 = pd.read_pickle(f"{p}/grid_part_3.pkl")[self.config.grid3_colnm]
        grid_df = pd.concat([grid_1, grid_2, grid_3], axis=1)
        del grid_1, grid_2, grid_3
        gc.collect()
        return grid_df

    def _prepare_data(self, base_grid_df, level, store, cat=None, dept=None):
        p = self.config.processed_data_dir

        grid_df = base_grid_df[base_grid_df['store_id'] == store]
        if level == 'store_cat':
            grid_df = grid_df[grid_df['cat_id'] == cat]
        elif level == 'store_dept':
            grid_df = grid_df[grid_df['dept_id'] == dept]

        grid_df = grid_df[grid_df['d'] >= self.config.first_day]

        lag = pd.read_pickle(f"{p}/lags_df_28.pkl")[self.config.lag_colnm]
        lag = lag[lag.index.isin(grid_df.index)]
        grid_df = pd.concat([grid_df, lag], axis=1)
        del lag
        gc.collect()

        mean_enc_cols = self.config.mean_enc_by_level[level]
        mean_enc = pd.read_pickle(f"{p}/mean_encoding_df.pkl")[mean_enc_cols]
        mean_enc = mean_enc[mean_enc.index.isin(grid_df.index)]
        grid_df = pd.concat([grid_df, mean_enc], axis=1)
        del mean_enc
        gc.collect()

        grid_df = reduce_mem_usage(grid_df)
        return grid_df

    def _train_one_group(self, base_grid_df, level, store, cat=None, dept=None):
        remove_feature = self.config.remove_feature_by_level[level]
        cv = self.config.cv

        grid_df = self._prepare_data(base_grid_df, level, store, cat=cat, dept=dept)
        model_var = grid_df.columns[~grid_df.columns.isin(remove_feature)]

        day_range = self.config.validation[cv]
        tr_mask = (grid_df['d'] <= day_range[0]) & (grid_df['d'] >= self.config.first_day)
        vl_mask = (grid_df['d'] > day_range[0]) & (grid_df['d'] <= day_range[1])

        train_data = lgb.Dataset(grid_df[tr_mask][model_var], label=grid_df[tr_mask]['sales'])
        valid_data = lgb.Dataset(grid_df[vl_mask][model_var], label=grid_df[vl_mask]['sales'])

        # verbose_eval was removed from lgb.train() in modern LightGBM (4.x) --
        # log_evaluation callback is the current equivalent.
        #
        # IMPORTANT: valid_sets must contain ONLY valid_data, never train_data.
        # Early stopping requires every tracked series to stop improving before
        # it halts -- training loss never stops improving by definition, so
        # including train_data here silently disables early stopping entirely
        # (this is why the previous run went the full 3000 rounds).
        m_lgb = lgb.train(
            self.config.lgb_params, train_data,
            valid_sets=[valid_data],
            valid_names=['valid'],
            callbacks=[
                lgb.log_evaluation(period=100),
                lgb.early_stopping(stopping_rounds=self.config.early_stopping_rounds),
            ]
        )

        if level == 'store':
            model_name = f"{self.config.models_dir}/non_recur_model_{store}.bin"
        elif level == 'store_cat':
            model_name = f"{self.config.models_dir}/non_recur_model_{store}_{cat}.bin"
        else:
            model_name = f"{self.config.models_dir}/non_recur_model_{store}_{dept}.bin"

        pickle.dump(m_lgb, open(model_name, 'wb'))
        logger.info(f"Saved {model_name}")

        del grid_df, train_data, valid_data, m_lgb, tr_mask, vl_mask
        gc.collect()

    def train_store_level(self):
        """A1's 2-1: one model per store (10 models)."""
        logger.info("Training non-recursive STORE-level models")
        base_grid_df = self._load_grid()
        for store in self.config.stores:
            logger.info(f"  {store}")
            self._train_one_group(base_grid_df, 'store', store)
        del base_grid_df
        gc.collect()

    def train_store_cat_level(self):
        """A1's 2-2: one model per store x category (30 models)."""
        logger.info("Training non-recursive STORE+CATEGORY-level models")
        base_grid_df = self._load_grid()
        for store in self.config.stores:
            for cat in self.config.cats:
                logger.info(f"  {store} / {cat}")
                self._train_one_group(base_grid_df, 'store_cat', store, cat=cat)
        del base_grid_df
        gc.collect()

    def train_store_dept_level(self):
        """A1's 2-3: one model per store x department (70 models)."""
        logger.info("Training non-recursive STORE+DEPARTMENT-level models")
        base_grid_df = self._load_grid()
        for store in self.config.stores:
            for dept in self.config.depts:
                logger.info(f"  {store} / {dept}")
                self._train_one_group(base_grid_df, 'store_dept', store, dept=dept)
        del base_grid_df
        gc.collect()

    def run(self):
        """Runs all 3 levels in sequence: 10 + 30 + 70 = 110 models total."""
        self.train_store_level()
        self.train_store_cat_level()
        self.train_store_dept_level()
        logger.info("All non-recursive training complete: 110 models saved")


In [11]:
try:
    config = ConfigurationManager()
    nonrecursive_trainer_config = config.get_nonrecursive_trainer_config()
    trainer = NonRecursiveTrainer(config=nonrecursive_trainer_config)
    trainer.run()
except Exception as e:
    raise e

[2026-09-23 16:04:12,573: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-23 16:04:12,602: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-23 16:04:12,613: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-23 16:04:12,615: INFO: common: created directory at: artifacts]
[2026-09-23 16:04:12,617: INFO: common: created directory at: artifacts/model_trainer/models]
[2026-09-23 16:04:12,618: INFO: 3854267898: Training non-recursive STORE-level models]
[2026-09-23 16:04:37,000: INFO: 3854267898:   CA_1]
Training until validation scores don't improve for 100 rounds
[100]	valid's rmse: 2.43842
Early stopping, best iteration is:
[1]	valid's rmse: 1.00307
[2026-09-23 16:05:34,455: INFO: 3854267898: Saved artifacts/model_trainer/models/non_recur_model_CA_1.bin]
[2026-09-23 16:05:34,567: INFO: 3854267898:   CA_2]
Training until validation scores don't improve for 100 rounds
[100]	valid's rmse: 2.11308
Early stopping, best iteration is:
